## The 20 newsgroup dataset

In [1]:
# choose, for example, 6 (or so) topics
categories = [
    'alt.atheism',
    'comp.windows.x',
    'rec.autos',
    'rec.sport.baseball',
    'sci.electronics',
    'sci.space'
]

In [2]:
from sklearn.datasets import fetch_20newsgroups

In [20]:
train = fetch_20newsgroups(subset='train',
                          categories=categories,
                          remove=('headers', 'footers', 'quotes'))
test = fetch_20newsgroups(subset='test',
                          categories=categories,
                          remove=('headers', 'footers', 'quotes'))

In [4]:
import pandas as pd

In [21]:
train_df = pd.DataFrame({'text':train['data'],
                         'category':train['target']})

In [12]:
test_df = pd.DataFrame({'text':test['data'],
                         'category':test['target']})

In [6]:
print(train_df.iloc[111].text)

I recently ftp'd Al's Circuit Simulator (ACS) and I'm looking for
the tutorial which is mentioned in the Users Manual (but not found there).
I don't have any experience constructing a netlist (such as for SPICE)
and I need a little help.

The examples which come with ACS aren't explanatory about the translation
between schematic and netlist. Does anyone have the fabled "Tutorial"
or any other reference which could help me in constructing a netlist from
a schematic diagram?

(I also emailed Al himself but received no response yet. He's probably
 busy with his next release.)
-- 
-------------------------------------------------------------------------------
Tim Pillsbury          Internet:  tsp@ccd.harris.com
                          uunet:  uunet!ccd.harris.com!timothy.pillsbury


My Plan:

- countvectorizer + naive bayes
- replace countvectorizer with tf-idf vectorizer
- stemming and lemmatization

tf - idf vectorizer

term frequency x inverse document frequency

$$
TF(w,d) = \frac{\text{number of times w appears in d}}{\text{total number of words in d}}
$$

$$
iDF(w,d) = \text{log}\frac{\text{total number of documents +1}}{\text{total number of documents containing w +1}}+1
$$

In [7]:
train['target_names']

['alt.atheism',
 'comp.windows.x',
 'rec.autos',
 'rec.sport.baseball',
 'sci.electronics',
 'sci.space']

In [24]:
train_df.category

0           atheism
1              cars
2           windows
3           atheism
4          baseball
           ...     
3443        windows
3444        windows
3445        atheism
3446       baseball
3447    electronics
Name: category, Length: 3448, dtype: object

In [23]:
# rename labels
new_names ={0:'atheism',
           1: 'windows',
           2: 'cars',
           3: 'baseball',
           4: 'electronics',
           5: 'space'}

train_df['category'] = train_df.category.map(new_names)
test_df['category'] = test_df.category.map(new_names)

In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [10]:
pipe = Pipeline(steps=[
    ('vect', TfidfVectorizer()),
    ('clf', MultinomialNB())
])
pipe

Pipeline(steps=[('vect', TfidfVectorizer()), ('clf', MultinomialNB())])

In [25]:
X_train = train_df.text
y_train = train_df.category
X_test = test_df.text
y_test = test_df.category

In [26]:
# grid search
param_dic = {
    'vect__max_features' : [1000,2000,5000,10000],
    'vect__stop_words': [None, 'english'],
    'vect__min_df': [1,5,20,50],
    'vect__max_df': [1.0,0.9,0.8],
    'vect__ngram_range': [(1,1),(1,2)],
    'vect__use_idf': [False, True]
}

grid = GridSearchCV(pipe, param_dic, cv=5, scoring='accuracy',n_jobs=-1)

In [27]:
y_train

0           atheism
1              cars
2           windows
3           atheism
4          baseball
           ...     
3443        windows
3444        windows
3445        atheism
3446       baseball
3447    electronics
Name: category, Length: 3448, dtype: object

In [28]:
grid.fit(X_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vect', TfidfVectorizer()),
                                       ('clf', MultinomialNB())]),
             n_jobs=-1,
             param_grid={'vect__max_df': [1.0, 0.9, 0.8],
                         'vect__max_features': [1000, 2000, 5000, 10000],
                         'vect__min_df': [1, 5, 20, 50],
                         'vect__ngram_range': [(1, 1), (1, 2)],
                         'vect__stop_words': [None, 'english'],
                         'vect__use_idf': [False, True]},
             scoring='accuracy')

In [30]:
grid.best_params_

{'vect__max_df': 1.0,
 'vect__max_features': 10000,
 'vect__min_df': 1,
 'vect__ngram_range': (1, 1),
 'vect__stop_words': 'english',
 'vect__use_idf': True}

In [31]:
best_pipe = grid.best_estimator_

In [32]:
y_train_pred = best_pipe.predict(X_train)
y_test_pred = best_pipe.predict(X_test)

In [33]:
accuracy_score(y_train, y_train_pred)

0.9504060324825986

In [34]:
accuracy_score(y_test, y_test_pred)

ValueError: Classification metrics can't handle a mix of unknown and multiclass targets

In [35]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred)

ValueError: Mix type of y not allowed, got types {'unknown', 'multiclass'}

In [37]:
best_pipe.predict(['I always wanted to be an astronaut',
                   'I hate Windows 10',
                   'I need to clean the windows of my car'])

array(['space', 'windows', 'cars'], dtype='<U11')